# 🔬 AI-Powered Victor Nivo Analysis for Bacterial Detection

**Complete workflow:** Design → Upload → Verify → Analyze → Explore

This notebook:
- Captures your experimental design (bacteria, phage, antibiotics, concentrations)
- Parses Victor Nivo data (OD, RLU, etc.)
- Finds early detection signals
- Lets you chat/verify before running
- Shows interactive visualizations

---

## 📦 Setup & Installation

In [ ]:
# Install dependencies
!pip install -q openpyxl plotly ipywidgets

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.signal import find_peaks
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from google.colab import files, drive
import warnings
warnings.filterwarnings('ignore')

# Plotting setup
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

print("✓ All dependencies installed")
print("✓ Ready to mount Drive or upload files")

## 💾 Mount Google Drive (Optional but Recommended)

Mount Drive to:
- Access your data files
- Save results automatically
- Keep analysis history

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Create analysis folder in Drive (if doesn't exist)
!mkdir -p '/content/drive/MyDrive/victor_nivo_analysis'
print("\n✓ Drive mounted at: /content/drive/MyDrive/")
print("✓ Analysis folder: /content/drive/MyDrive/victor_nivo_analysis/")

## 📄 Upload or Select Data File

Choose one option:

In [ ]:
# OPTION 1: Upload file from your computer
def upload_data_file():
    print("Click 'Choose Files' and select your Victor Nivo Excel file...")
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        print(f"\n✓ Uploaded: {filename}")
        return filename
    return None

# OPTION 2: Use file from Drive (uncomment and edit path)
# data_file = '/content/drive/MyDrive/victor_nivo_analysis/my_experiment.xlsx'

# Run one of these:
data_file = upload_data_file()  # Upload
# data_file = '/content/drive/MyDrive/...'  # Or specify Drive path

## 🧬 EXPERIMENTAL DESIGN BUILDER

**This is the key part!** Tell the system what variables you're testing.

### Step 1: Define Your Variables

In [ ]:
# Define what you're testing in this experiment
# Add/remove variables as needed for YOUR experiment

experimental_variables = {
    'bacteria_strain': [
        'E.coli_O157H7',
        'E.coli_K12',
        'Salmonella',
        'None'  # For blanks
    ],
    
    'bacteria_concentration': [
        '1e8',
        '1e7',
        '1e6',
        '1e5',
        '1e4',
        '0'  # For negative controls
    ],
    
    'phage_present': [
        'Yes',
        'No'
    ],
    
    'phage_titer': [
        '1e9',
        '1e8',
        '1e7',
        '0'
    ],
    
    'antibiotic': [
        'None',
        'Ampicillin',
        'Kanamycin',
        'Gentamicin'
    ],
    
    'antibiotic_concentration': [
        '0',
        '10ug/mL',
        '50ug/mL',
        '100ug/mL'
    ],
    
    'salt_concentration': [
        '0M',
        '0.5M',
        '1.0M'
    ],
    
    'sample_type': [
        'Test',
        'Positive_Control',
        'Negative_Control',
        'Blank'
    ]
}

print("✓ Experimental variables defined")
print(f"\nVariables you're tracking: {list(experimental_variables.keys())}")
print("\nEdit the cell above to match YOUR experiment!")

### Step 2: Interactive Plate Layout Builder

Click on wells to assign conditions:

In [ ]:
# Interactive plate layout tool
plate_design = {}

def create_plate_designer():
    """Interactive widget to design plate layout."""
    
    # Well selector
    well_selector = widgets.Dropdown(
        options=['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12',
                 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12',
                 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12',
                 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12',
                 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12',
                 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12',
                 'G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12',
                 'H1', 'H2', 'H3', 'H4', 'H5', 'H6', 'H7', 'H8', 'H9', 'H10', 'H11', 'H12'],
        description='Well:'
    )
    
    # Variable inputs
    variable_inputs = {}
    for var_name, var_options in experimental_variables.items():
        variable_inputs[var_name] = widgets.Dropdown(
            options=var_options,
            description=var_name.replace('_', ' ').title() + ':',
            style={'description_width': '200px'}
        )
    
    # Add button
    add_button = widgets.Button(description='Add Well', button_style='success')
    status_label = widgets.Label()
    
    def on_add_click(b):
        well = well_selector.value
        plate_design[well] = {var: inp.value for var, inp in variable_inputs.items()}
        status_label.value = f"✓ Added {well}: {plate_design[well]}"
    
    add_button.on_click(on_add_click)
    
    # Layout
    inputs_box = widgets.VBox([well_selector] + list(variable_inputs.values()) + [add_button, status_label])
    
    display(inputs_box)
    
    return plate_design

print("Interactive Plate Designer:")
print("Select a well and fill in the conditions, then click 'Add Well'")
print("Repeat for all wells in your experiment.\n")

plate_design = create_plate_designer()

### Alternative: Quick Batch Fill

For systematic plate layouts (e.g., rows = strains, columns = concentrations):

In [ ]:
# Example: Batch fill plate design
# Uncomment and customize for your layout pattern

# Example 1: Row A = O157H7 dilution series, Row B = K12 dilution series
'''
concentrations = ['1e8', '1e7', '1e6', '1e5', '1e4', '1e3', '1e2', '1e1', '0', '0', '0', '0']

for col in range(1, 13):
    # Row A: E. coli O157:H7 with phage
    plate_design[f'A{col}'] = {
        'bacteria_strain': 'E.coli_O157H7',
        'bacteria_concentration': concentrations[col-1],
        'phage_present': 'Yes',
        'phage_titer': '1e8',
        'antibiotic': 'None',
        'antibiotic_concentration': '0',
        'salt_concentration': '0M',
        'sample_type': 'Test'
    }
    
    # Row B: E. coli K12 with phage
    plate_design[f'B{col}'] = {
        'bacteria_strain': 'E.coli_K12',
        'bacteria_concentration': concentrations[col-1],
        'phage_present': 'Yes',
        'phage_titer': '1e8',
        'antibiotic': 'None',
        'antibiotic_concentration': '0',
        'salt_concentration': '0M',
        'sample_type': 'Test'
    }

print(f"✓ Batch filled {len(plate_design)} wells")
'''

# Or load from CSV file
# plate_design_df = pd.read_csv('my_plate_layout.csv')
# plate_design = plate_design_df.set_index('Well').to_dict('index')

print(f"Current plate design has {len(plate_design)} wells defined")

### View Your Plate Design

In [ ]:
# Convert to DataFrame for easy viewing
if plate_design:
    design_df = pd.DataFrame(plate_design).T
    design_df.index.name = 'Well'
    
    print("=" * 80)
    print("YOUR EXPERIMENTAL DESIGN")
    print("=" * 80)
    display(design_df)
    
    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    for var in experimental_variables.keys():
        if var in design_df.columns:
            print(f"\n{var.replace('_', ' ').title()}:")
            print(design_df[var].value_counts().to_string())
else:
    print("⚠ No wells defined yet. Use the plate designer above.")

## 💬 CHAT & VERIFY

**Before running the analysis, let's verify the design makes sense!**

In [ ]:
# Analysis of your experimental design
def verify_experimental_design(design_df):
    """Check if the experimental design is sensible."""
    
    issues = []
    recommendations = []
    
    print("🔍 DESIGN VERIFICATION\n" + "=" * 80)
    
    # Check 1: Do we have controls?
    if 'sample_type' in design_df.columns:
        has_blank = (design_df['sample_type'] == 'Blank').any()
        has_neg_control = (design_df['sample_type'] == 'Negative_Control').any()
        has_pos_control = (design_df['sample_type'] == 'Positive_Control').any()
        
        if not has_blank:
            issues.append("❌ No blank wells found. Add media-only controls!")
        else:
            print("✓ Blank controls present")
        
        if not has_neg_control:
            recommendations.append("💡 Consider adding negative controls (no bacteria)")
        else:
            print("✓ Negative controls present")
    
    # Check 2: Do we have replicates?
    if len(design_df) > 0:
        # Count unique conditions
        condition_cols = [c for c in design_df.columns if c != 'sample_type']
        unique_conditions = design_df[condition_cols].drop_duplicates()
        
        total_wells = len(design_df)
        unique_conds = len(unique_conditions)
        
        if total_wells == unique_conds:
            issues.append("❌ No replicates! Each condition only tested once.")
        elif total_wells / unique_conds < 2:
            recommendations.append(f"💡 Limited replicates (avg {total_wells/unique_conds:.1f} per condition)")
        else:
            print(f"✓ Good replication (avg {total_wells/unique_conds:.1f} wells per condition)")
    
    # Check 3: Concentration series?
    if 'bacteria_concentration' in design_df.columns:
        unique_concs = design_df['bacteria_concentration'].nunique()
        if unique_concs >= 4:
            print(f"✓ Good concentration range ({unique_concs} levels)")
        elif unique_concs > 1:
            recommendations.append(f"💡 Limited concentration range ({unique_concs} levels)")
    
    # Check 4: Variable coverage
    print("\n📊 VARIABLE COVERAGE:")
    for col in design_df.columns:
        unique_vals = design_df[col].nunique()
        print(f"  {col}: {unique_vals} different values")
        if unique_vals == 1:
            recommendations.append(f"💡 '{col}' is constant (not being tested)")
    
    # Summary
    print("\n" + "=" * 80)
    if issues:
        print("\n⚠️  ISSUES TO ADDRESS:")
        for issue in issues:
            print(f"  {issue}")
    
    if recommendations:
        print("\n💡 RECOMMENDATIONS:")
        for rec in recommendations:
            print(f"  {rec}")
    
    if not issues:
        print("\n✅ Design looks good! Ready to analyze.")
    
    return len(issues) == 0

if plate_design:
    design_ok = verify_experimental_design(design_df)
else:
    print("⚠ Define your plate design first!")

### Chat About Your Design

Ask questions before proceeding:

In [ ]:
# Interactive Q&A about your design

def answer_design_questions(design_df, question):
    """Answer common questions about the experimental design."""
    
    q_lower = question.lower()
    
    # Question patterns
    if 'how many' in q_lower:
        if 'replicates' in q_lower or 'wells per' in q_lower:
            # Count replicates per condition
            condition_cols = [c for c in design_df.columns if c != 'sample_type']
            replicate_counts = design_df.groupby(condition_cols).size()
            print(f"Replicates per condition:")
            print(f"  Min: {replicate_counts.min()}")
            print(f"  Max: {replicate_counts.max()}")
            print(f"  Mean: {replicate_counts.mean():.1f}")
        
        elif 'controls' in q_lower:
            if 'sample_type' in design_df.columns:
                print(design_df['sample_type'].value_counts())
        
        else:
            print(f"Total wells: {len(design_df)}")
    
    elif 'which wells' in q_lower or 'what wells' in q_lower:
        # Find wells matching criteria
        if 'phage' in q_lower:
            phage_wells = design_df[design_df['phage_present'] == 'Yes'].index.tolist()
            print(f"Wells with phage: {phage_wells}")
        
        elif 'antibiotic' in q_lower:
            abx_wells = design_df[design_df['antibiotic'] != 'None'].index.tolist()
            print(f"Wells with antibiotics: {abx_wells}")
        
        elif 'o157' in q_lower:
            o157_wells = design_df[design_df['bacteria_strain'].str.contains('O157', na=False)].index.tolist()
            print(f"Wells with O157:H7: {o157_wells}")
    
    elif 'compare' in q_lower or 'difference' in q_lower:
        print("For comparisons, I'll need to run the full analysis.")
        print("But I can tell you what variables are being tested:")
        for col in design_df.columns:
            if design_df[col].nunique() > 1:
                print(f"  - {col}: {design_df[col].nunique()} levels")
    
    elif 'missing' in q_lower or 'forgot' in q_lower:
        # Check for missing controls or conditions
        print("Checking for potential gaps...")
        if 'bacteria_concentration' in design_df.columns:
            if not (design_df['bacteria_concentration'] == '0').any():
                print("  ⚠️ No zero-bacteria controls")
        if 'phage_present' in design_df.columns:
            if design_df['phage_present'].nunique() == 1:
                print("  ⚠️ All wells have same phage status (no comparison possible)")
    
    else:
        print("I can answer questions like:")
        print("  - 'How many replicates per condition?'")
        print("  - 'Which wells have phage?'")
        print("  - 'How many controls do I have?'")
        print("  - 'Am I missing anything?'")

# Try asking questions:
if plate_design:
    print("Ask me about your design:")
    print("(Edit the question in the next cell)\n")
else:
    print("Define your plate design first!")

In [ ]:
# Ask questions here
question = "How many replicates per condition?"  # Edit this!

if plate_design:
    answer_design_questions(design_df, question)

## 🔧 [All the parser and analyzer code goes here - embedded]

I'll add all the parsing and analysis code in collapsed cells...

In [ ]:
#@title Victor Nivo Parser (Click to expand)

class MultiOperationVictorNivoParser:
    """Parse Victor Nivo files with multiple measurement operations."""
    
    def __init__(self, filepath, operation_labels=None):
        self.filepath = Path(filepath)
        self.raw_data = pd.read_excel(filepath, sheet_name=0, header=None)
        self.operation_labels = operation_labels or {}
        
    def parse(self):
        """Parse file and return wide-format dataframe."""
        print(f"\nParsing: {self.filepath.name}")
        print("=" * 80)
        
        data_sections = self._find_data_sections()
        all_data = {}
        
        for op_num, section_info in data_sections.items():
            df = self._parse_section(op_num, section_info)
            measurement_type = self.operation_labels.get(op_num, f"Op{op_num}")
            all_data[measurement_type] = df
            print(f"  ✓ {measurement_type}: {len(df)} wells × {len(df.columns)-1} timepoints")
        
        consolidated = self._merge_measurements(all_data)
        print(f"\n✓ Consolidated: {consolidated.shape[0]} wells, {consolidated.shape[1]} columns")
        
        return consolidated
    
    def _find_data_sections(self):
        sections = {}
        for i in range(len(self.raw_data)):
            if self.raw_data.iloc[i, 0] == 'Well':
                for j in range(max(0, i-15), i):
                    cell = str(self.raw_data.iloc[j, 0])
                    if 'OPERATION' in cell:
                        op_num = cell.split()[0]
                        temp = None
                        for k in range(j, i):
                            if self.raw_data.iloc[k, 0] == 'TEMPERATURE(Celsius)':
                                temp = self.raw_data.iloc[k, 1]
                                break
                        sections[op_num] = {'well_row': i, 'temperature': temp}
                        break
        return sections
    
    def _parse_section(self, op_num, section_info):
        well_row = section_info['well_row']
        times = []
        
        for col_idx in range(2, len(self.raw_data.columns)):
            time_val = self.raw_data.iloc[well_row, col_idx]
            if pd.notna(time_val):
                try:
                    times.append(float(time_val))
                except (ValueError, TypeError):
                    break
            else:
                break
        
        data_rows = []
        row_idx = well_row + 1
        
        while row_idx < len(self.raw_data):
            well_name = self.raw_data.iloc[row_idx, 0]
            if pd.notna(well_name) and isinstance(well_name, str) and len(str(well_name)) <= 3:
                values = []
                for col_idx in range(2, 2 + len(times)):
                    val = self.raw_data.iloc[row_idx, col_idx]
                    if pd.notna(val):
                        try:
                            values.append(float(val))
                        except (ValueError, TypeError):
                            values.append(np.nan)
                    else:
                        values.append(np.nan)
                
                row_data = {'Well': well_name}
                for time_idx, val in enumerate(values):
                    row_data[f'T{time_idx}_sec'] = val
                
                data_rows.append(row_data)
                row_idx += 1
            else:
                break
        
        df = pd.DataFrame(data_rows)
        df.attrs['times'] = times
        df.attrs['temperature'] = section_info['temperature']
        return df
    
    def _merge_measurements(self, all_data):
        base_df = None
        
        for measurement_type, df in all_data.items():
            if base_df is None:
                base_df = df[['Well']].copy()
            
            renamed = df.copy()
            for col in df.columns:
                if col != 'Well':
                    renamed.rename(columns={col: f'{measurement_type}_{col}'}, inplace=True)
            
            base_df = base_df.merge(renamed, on='Well', how='outer')
        
        first_measurement = list(all_data.values())[0]
        times = first_measurement.attrs['times']
        
        for idx, time_sec in enumerate(times):
            base_df[f'Time_{idx}_seconds'] = time_sec
            base_df[f'Time_{idx}_hours'] = time_sec / 3600
        
        return base_df

def quick_parse(filepath, **operation_labels):
    labels = {k.replace('Op', ''): v for k, v in operation_labels.items()}
    parser = MultiOperationVictorNivoParser(filepath, labels)
    return parser.parse()

print("✓ Parser code loaded")

## 📊 Parse Your Data

Tell the system which operations are which measurements:

In [ ]:
# Define measurement types
# Check your Victor Nivo export to see which operation numbers correspond to which measurements

measurement_operations = {
    'Op4': 'OD600',    # Operation 4 = OD at 600nm
    'Op5': 'OD560',    # Operation 5 = OD at 560nm
    'Op6': 'OD450',    # Operation 6 = OD at 450nm
    # 'Op7': 'RLU',    # Uncomment if you have bioluminescence
}

# Parse the data
if data_file:
    parsed_df = quick_parse(data_file, **measurement_operations)
    
    print("\n" + "=" * 80)
    print("PARSED DATA PREVIEW")
    print("=" * 80)
    display(parsed_df.head())
    
    # Save parsed data
    output_file = '/content/drive/MyDrive/victor_nivo_analysis/parsed_data.csv'
    parsed_df.to_csv(output_file, index=False)
    print(f"\n✓ Saved to: {output_file}")
else:
    print("⚠ Upload or specify data file first!")

## 🔗 Merge Experimental Design with Data

In [ ]:
# Merge plate design with parsed data
if plate_design and 'parsed_df' in locals():
    # Convert plate design to DataFrame
    design_df = pd.DataFrame(plate_design).T
    design_df.index.name = 'Well'
    design_df = design_df.reset_index()
    
    # Merge
    full_data = parsed_df.merge(design_df, on='Well', how='left')
    
    print("✓ Merged experimental design with measurement data")
    print(f"\nData shape: {full_data.shape}")
    print(f"Columns: {list(full_data.columns[:10])}...")
    
    # Check for unlabeled wells
    unlabeled = full_data[full_data['bacteria_strain'].isna()]['Well'].tolist()
    if unlabeled:
        print(f"\n⚠️ Warning: {len(unlabeled)} wells not in experimental design: {unlabeled[:10]}...")
else:
    print("⚠ Need both data and plate design!")
    full_data = parsed_df

## 📈 Quick Visual Check

Let's look at the raw data before analyzing:

In [ ]:
# Plot growth curves colored by experimental variable
if 'full_data' in locals():
    
    def plot_by_variable(data, measurement='OD600', color_by='bacteria_strain'):
        """Plot growth curves colored by an experimental variable."""
        
        # Get time points
        time_cols = [c for c in data.columns if 'Time_' in c and '_hours' in c]
        times = [data[c].iloc[0] for c in time_cols]
        
        plt.figure(figsize=(16, 8))
        
        # Get unique values for color coding
        if color_by in data.columns:
            unique_vals = data[color_by].dropna().unique()
            colors = plt.cm.tab10(np.linspace(0, 1, len(unique_vals)))
            color_map = dict(zip(unique_vals, colors))
            
            for idx, row in data.iterrows():
                well = row['Well']
                color_val = row.get(color_by, 'Unknown')
                color = color_map.get(color_val, 'gray')
                
                # Extract values
                values = []
                for t_idx in range(len(times)):
                    col_name = f'{measurement}_T{t_idx}_sec'
                    if col_name in row.index:
                        values.append(row[col_name])
                
                if len(values) == len(times):
                    plt.plot(times, values, color=color, alpha=0.6, linewidth=1.5)
            
            # Legend
            handles = [plt.Line2D([0], [0], color=color_map[val], linewidth=3, label=val) 
                      for val in unique_vals]
            plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
        
        else:
            # No color coding
            for idx, row in data.iterrows():
                well = row['Well']
                values = []
                for t_idx in range(len(times)):
                    col_name = f'{measurement}_T{t_idx}_sec'
                    if col_name in row.index:
                        values.append(row[col_name])
                
                if len(values) == len(times):
                    plt.plot(times, values, alpha=0.5, linewidth=1)
        
        plt.xlabel('Time (hours)', fontsize=14)
        plt.ylabel(measurement, fontsize=14)
        plt.title(f'{measurement} Growth Curves (colored by {color_by})', fontsize=16)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # Plot OD600 by bacterial strain
    plot_by_variable(full_data, 'OD600', 'bacteria_strain')
    
    # Plot OD600 by phage status
    if 'phage_present' in full_data.columns:
        plot_by_variable(full_data, 'OD600', 'phage_present')

else:
    print("⚠ Parse data first!")

### Does the data look reasonable?

Check:
- ✅ Blanks are flat (near zero)
- ✅ Positive controls show growth
- ✅ Replicates cluster together
- ✅ No obvious outliers or failed wells

If something looks wrong, go back and fix the plate design or check your data file!

## 🤖 RUN ANALYSIS

[Rest of the analysis code - feature generation, early detection, etc.]

## 🤖 Feature Generation Engine

Generate features that understand your experimental variables:

In [ ]:
#@title Advanced Feature Generator (Click to expand code)

class VariableAwareFeatureGenerator:
    """Generate features that understand experimental variables."""
    
    def __init__(self, data_df, experimental_design):
        self.data = data_df
        self.design = experimental_design
        
    def generate_all_features(self):
        """Generate comprehensive feature set."""
        print("Generating features...")
        features_list = []
        
        # Get measurement types
        measurement_types = self._get_measurement_types()
        n_timepoints = self._get_n_timepoints()
        
        for idx, row in self.data.iterrows():
            well = row['Well']
            feature_row = {'Well': well}
            
            # Add experimental variables
            if well in self.design.index:
                for var in self.design.columns:
                    feature_row[var] = self.design.loc[well, var]
            
            # Generate features for each measurement type
            for mtype in measurement_types:
                values = self._get_timeseries(row, mtype, n_timepoints)
                
                if len(values) < 5:
                    continue
                
                # Basic stats
                feature_row[f'{mtype}_mean'] = np.mean(values)
                feature_row[f'{mtype}_max'] = np.max(values)
                feature_row[f'{mtype}_final'] = values[-1]
                
                # Early timepoint features
                for early_n in [3, 5, 10, 15]:
                    if early_n < len(values):
                        early_vals = values[:early_n]
                        feature_row[f'{mtype}_early{early_n}_mean'] = np.mean(early_vals)
                        feature_row[f'{mtype}_early{early_n}_slope'] = self._slope(early_vals)
                
                # Growth characteristics
                deriv = np.diff(values)
                if len(deriv) > 0:
                    feature_row[f'{mtype}_max_growth_rate'] = np.max(deriv)
                    feature_row[f'{mtype}_time_to_max_rate'] = np.argmax(deriv)
                
                # Lag detection
                baseline = np.mean(values[:min(5, len(values))])
                for factor in [1.1, 1.2, 1.5, 2.0]:
                    threshold = baseline * factor
                    exceed = np.where(values > threshold)[0]
                    if len(exceed) > 0:
                        feature_row[f'{mtype}_time_to_{int(factor*10)}x'] = exceed[0]
                    else:
                        feature_row[f'{mtype}_time_to_{int(factor*10)}x'] = n_timepoints
            
            # Cross-measurement features
            if len(measurement_types) >= 2:
                for i, m1 in enumerate(measurement_types):
                    for m2 in measurement_types[i+1:]:
                        v1 = self._get_timeseries(row, m1, n_timepoints)
                        v2 = self._get_timeseries(row, m2, n_timepoints)
                        
                        min_len = min(len(v1), len(v2))
                        if min_len > 3:
                            v1, v2 = v1[:min_len], v2[:min_len]
                            
                            # Ratio
                            with np.errstate(divide='ignore', invalid='ignore'):
                                ratio = v1 / v2
                                ratio = ratio[np.isfinite(ratio)]
                                if len(ratio) > 0:
                                    feature_row[f'{m1}_div_{m2}_mean'] = np.mean(ratio)
                                    feature_row[f'{m1}_div_{m2}_early'] = np.mean(ratio[:min(5, len(ratio))])
            
            features_list.append(feature_row)
        
        return pd.DataFrame(features_list)
    
    def _get_measurement_types(self):
        types = set()
        for col in self.data.columns:
            if '_T' in col and '_sec' in col:
                types.add(col.split('_T')[0])
        return sorted(types)
    
    def _get_n_timepoints(self):
        time_cols = [c for c in self.data.columns if 'Time_' in c and '_seconds' in c]
        return len(time_cols)
    
    def _get_timeseries(self, row, mtype, n_timepoints):
        values = []
        for t in range(n_timepoints):
            col = f'{mtype}_T{t}_sec'
            if col in row.index and pd.notna(row[col]):
                values.append(float(row[col]))
        return np.array(values)
    
    def _slope(self, values):
        if len(values) < 2:
            return 0
        x = np.arange(len(values))
        return np.polyfit(x, values, 1)[0]

print("✓ Feature generator loaded")

## 🔬 Run Feature Generation

In [ ]:
if 'full_data' in locals() and plate_design:
    generator = VariableAwareFeatureGenerator(full_data, design_df.set_index('Well'))
    features_df = generator.generate_all_features()
    
    print(f"\n✓ Generated {len(features_df.columns)} features for {len(features_df)} wells")
    
    # Show sample
    display(features_df.head())
    
    # Save
    features_df.to_csv('/content/drive/MyDrive/victor_nivo_analysis/features.csv', index=False)
    print("\n✓ Saved features to Drive")
else:
    print("⚠ Need data and experimental design first!")

## 🎯 Find Early Detection Signals

Which features discriminate between your experimental conditions?

In [ ]:
def find_discriminative_features(features_df, compare_variable='bacteria_strain'):
    """Find features that discriminate between groups."""
    
    if compare_variable not in features_df.columns:
        print(f"⚠ Variable '{compare_variable}' not found")
        return None
    
    results = []
    
    # Get groups
    groups = features_df[compare_variable].dropna().unique()
    
    if len(groups) < 2:
        print("⚠ Need at least 2 groups to compare")
        return None
    
    print(f"Comparing groups: {list(groups)}")
    
    # Test each feature
    feature_cols = [c for c in features_df.columns 
                   if c not in ['Well'] + list(experimental_variables.keys())]
    
    for col in feature_cols:
        # Early features are most interesting
        is_early = any(x in col for x in ['early', 'time_to'])
        
        # Get values per group
        group_values = []
        for group in groups:
            vals = features_df[features_df[compare_variable] == group][col].dropna()
            if len(vals) > 1:
                group_values.append(vals.values)
        
        if len(group_values) >= 2:
            try:
                # T-test between first two groups
                stat, pval = stats.ttest_ind(group_values[0], group_values[1], equal_var=False)
                
                # Effect size (Cohen's d)
                mean_diff = abs(np.mean(group_values[0]) - np.mean(group_values[1]))
                pooled_std = np.sqrt((np.var(group_values[0]) + np.var(group_values[1])) / 2)
                effect_size = mean_diff / pooled_std if pooled_std > 0 else 0
                
                if pval < 0.05:  # Significant
                    results.append({
                        'feature': col,
                        'p_value': pval,
                        'effect_size': effect_size,
                        'group1_mean': np.mean(group_values[0]),
                        'group2_mean': np.mean(group_values[1]),
                        'is_early': is_early
                    })
            except:
                pass
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('effect_size', ascending=False)
    
    return results_df

# Run analysis
if 'features_df' in locals():
    # Choose which variable to compare
    compare_var = 'bacteria_strain'  # Change this to compare different variables!
    
    signals_df = find_discriminative_features(features_df, compare_var)
    
    if signals_df is not None and len(signals_df) > 0:
        print(f"\n{'='*80}")
        print(f"EARLY DETECTION SIGNALS (comparing {compare_var})")
        print('='*80)
        
        # Show top early signals
        early_signals = signals_df[signals_df['is_early'] == True].head(10)
        
        for i, row in early_signals.iterrows():
            print(f"\n{i+1}. {row['feature']}")
            print(f"   Effect size: {row['effect_size']:.2f}")
            print(f"   P-value: {row['p_value']:.2e}")
            print(f"   Group means: {row['group1_mean']:.3f} vs {row['group2_mean']:.3f}")
        
        # Save
        signals_df.to_csv('/content/drive/MyDrive/victor_nivo_analysis/detection_signals.csv', index=False)
        print("\n✓ Saved to Drive")
else:
    print("⚠ Generate features first!")

## 🔍 Interaction Analysis

Does phage + antibiotic show synergy? Do certain combinations work better?

In [ ]:
def analyze_interactions(features_df, var1='phage_present', var2='antibiotic', 
                        outcome='OD600_time_to_20x'):
    """Analyze interactions between two experimental variables."""
    
    if var1 not in features_df.columns or var2 not in features_df.columns:
        print(f"⚠ Variables not found in data")
        return None
    
    if outcome not in features_df.columns:
        # Try to find a similar outcome
        possible = [c for c in features_df.columns if 'time_to' in c]
        if possible:
            outcome = possible[0]
            print(f"Using outcome: {outcome}")
        else:
            print("⚠ No suitable outcome found")
            return None
    
    # Create interaction groups
    interaction_data = []
    
    for val1 in features_df[var1].dropna().unique():
        for val2 in features_df[var2].dropna().unique():
            mask = (features_df[var1] == val1) & (features_df[var2] == val2)
            values = features_df[mask][outcome].dropna()
            
            if len(values) > 0:
                interaction_data.append({
                    var1: val1,
                    var2: val2,
                    'mean': np.mean(values),
                    'std': np.std(values),
                    'n': len(values)
                })
    
    interaction_df = pd.DataFrame(interaction_data)
    
    # Visualize
    if len(interaction_df) > 0:
        plt.figure(figsize=(10, 6))
        
        # Pivot for heatmap
        pivot = interaction_df.pivot(index=var1, columns=var2, values='mean')
        
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r')
        plt.title(f'Interaction Effect: {var1} × {var2}\n(Outcome: {outcome})', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        print("\nInteraction Results:")
        display(interaction_df)
        
        # Interpret
        print("\n💡 INTERPRETATION:")
        print(f"Lower {outcome} = earlier detection (better)")
        best = interaction_df.loc[interaction_df['mean'].idxmin()]
        print(f"\nBest combination: {best[var1]} + {best[var2]}")
        print(f"Detection time: {best['mean']:.1f} timepoints")
    
    return interaction_df

# Run interaction analysis
if 'features_df' in locals():
    # Analyze phage + antibiotic interaction
    if 'phage_present' in features_df.columns and 'antibiotic' in features_df.columns:
        print("Analyzing Phage × Antibiotic Interaction...\n")
        interact_df = analyze_interactions(features_df, 'phage_present', 'antibiotic')
    else:
        print("Define phage_present and antibiotic in your experimental design to see interactions")

## 📊 Interactive Results Explorer

Explore your results interactively:

In [ ]:
# Interactive dropdowns to explore results
if 'features_df' in locals():
    
    # Variable selector
    exp_vars = [v for v in experimental_variables.keys() if v in features_df.columns]
    
    compare_dropdown = widgets.Dropdown(
        options=exp_vars,
        description='Compare:',
        value=exp_vars[0] if exp_vars else None
    )
    
    outcome_dropdown = widgets.Dropdown(
        options=[c for c in features_df.columns if 'time_to' in c or 'early' in c],
        description='Outcome:',
    )
    
    analyze_button = widgets.Button(
        description='Analyze',
        button_style='success'
    )
    
    output_area = widgets.Output()
    
    def on_analyze(b):
        with output_area:
            output_area.clear_output()
            
            compare_var = compare_dropdown.value
            outcome_var = outcome_dropdown.value
            
            # Group by variable and show means
            grouped = features_df.groupby(compare_var)[outcome_var].agg(['mean', 'std', 'count'])
            
            print(f"\n{outcome_var} by {compare_var}:")
            display(grouped)
            
            # Visualize
            plt.figure(figsize=(10, 6))
            grouped['mean'].plot(kind='bar', yerr=grouped['std'])
            plt.title(f'{outcome_var} by {compare_var}')
            plt.ylabel(outcome_var)
            plt.xlabel(compare_var)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
    
    analyze_button.on_click(on_analyze)
    
    display(widgets.VBox([compare_dropdown, outcome_dropdown, analyze_button, output_area]))
else:
    print("⚠ Generate features first!")

## 💾 Download All Results

Package everything for download:

In [ ]:
# Create comprehensive results package
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'/content/results_{timestamp}'
!mkdir -p {results_dir}

# Save all results
if 'parsed_df' in locals():
    parsed_df.to_csv(f'{results_dir}/parsed_data.csv', index=False)
    print("✓ Saved parsed data")

if 'design_df' in locals():
    design_df.to_csv(f'{results_dir}/experimental_design.csv', index=False)
    print("✓ Saved experimental design")

if 'features_df' in locals():
    features_df.to_csv(f'{results_dir}/features.csv', index=False)
    print("✓ Saved features")

if 'signals_df' in locals():
    signals_df.to_csv(f'{results_dir}/detection_signals.csv', index=False)
    print("✓ Saved detection signals")

if 'interact_df' in locals():
    interact_df.to_csv(f'{results_dir}/interactions.csv', index=False)
    print("✓ Saved interaction analysis")

# Create summary report
with open(f'{results_dir}/SUMMARY.txt', 'w') as f:
    f.write("VICTOR NIVO ANALYSIS RESULTS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Data File: {data_file if 'data_file' in locals() else 'Unknown'}\n\n")
    
    if 'parsed_df' in locals():
        f.write(f"Wells Analyzed: {len(parsed_df)}\n")
    
    if 'features_df' in locals():
        f.write(f"Features Generated: {len(features_df.columns)}\n")
    
    if 'signals_df' in locals():
        f.write(f"\nSignificant Signals Found: {len(signals_df)}\n")
        f.write(f"\nTop 5 Early Detection Signals:\n")
        for i, row in signals_df.head(5).iterrows():
            f.write(f"\n{i+1}. {row['feature']}\n")
            f.write(f"   Effect size: {row['effect_size']:.2f}\n")
            f.write(f"   P-value: {row['p_value']:.2e}\n")

print("\n✓ Created summary report")

# Zip everything
zip_path = f'/content/victor_nivo_results_{timestamp}.zip'
!cd /content && zip -r {zip_path} {results_dir}

print(f"\n{'='*80}")
print("RESULTS PACKAGE READY")
print('='*80)
print(f"\nDownload: {zip_path}")
print("\nOr find in Drive: /MyDrive/victor_nivo_analysis/")

# Copy to Drive
!cp {zip_path} /content/drive/MyDrive/victor_nivo_analysis/
print("\n✓ Copied to Google Drive")

# Download button
files.download(zip_path)

## 🎉 You're Done!

### What You've Accomplished:

1. ✅ Defined your experimental variables
2. ✅ Labeled your plate layout
3. ✅ Verified your design made sense
4. ✅ Visualized your raw data
5. ✅ Generated 100+ features per well
6. ✅ Found early detection signals
7. ✅ Analyzed variable interactions
8. ✅ Downloaded complete results

### Next Steps:

- Review the detection signals CSV to see best early indicators
- Check interaction analysis for synergies
- Run this notebook on your next experiment!
- Modify experimental variables as needed

### Questions?

Re-run any cell to try different analyses. The notebook saves everything to Drive automatically.

---

**Built for phage-based pathogen detection research 🦠**